In [33]:
import duckdb

con = duckdb.connect()

path = "../../source_database/seed_data/supplylink_dc_movements.csv"
suppliers_path = "../../source_database/seed_data/suppliers.csv"

raw = f"read_csv_auto('{path}', types={{'movement_date': 'VARCHAR'}})"

In [6]:
con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").show()

┌───────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name  │ column_type │  null   │   key   │ default │  extra  │
│    varchar    │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ movement_id   │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ dc_code       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ sku           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ movement_type │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ quantity      │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ supplier_ref  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ movement_date │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ movement_time │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ timezone      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ reached_epos  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NU

In [3]:
# ── Completeness Audit: how many NULLs does each column have? ──

# Step 1: Get the list of column names automatically from the file.
# This way the same audit code works on ANY csv
columns_df = con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").df()
column_names = columns_df['column_name'].tolist()

# Step 2: Get the total row count once, so we can turn null counts into %.
total_rows = con.sql(f"SELECT COUNT(*) AS cnt FROM read_csv_auto('{path}')").df()['cnt'][0]
print(f"Total rows in file: {total_rows}")

# Step 3: For each column, ask DuckDB:
#   COUNT(*)        -> counts every row (NULL or not)
#   COUNT(col_name) -> counts only NON-null rows for that column
#   so COUNT(*) - COUNT(col_name) = how many rows are NULL in that column
null_report = []
for col in column_names:
    result = con.sql(f"""
        SELECT
            '{col}' AS column_name,
            COUNT(*) - COUNT({col}) AS null_count,
            ROUND(100.0 * (COUNT(*) - COUNT({col})) / COUNT(*), 2) AS null_percentage
        FROM read_csv_auto('{path}')
    """).df()
    null_report.append(result)

# Step 4: Stack all the per-column results into one table and show it.
import pandas as pd
null_report_df = pd.concat(null_report, ignore_index=True)
null_report_df

Total rows in file: 11171


,column_name,null_count,null_percentage
0,movement_id,0,0.00
1,dc_code,0,0.00
2,sku,0,0.00
3,movement_type,0,0.00
4,quantity,0,0.00
5,supplier_ref,0,0.00
6,movement_date,0,0.00
7,movement_time,0,0.00
8,timezone,0,0.00
9,reached_epos,101,0.90


In [7]:
# Uniqueness

# WHY: movement_id is the primary key. Also checking for a near-duplicate
# delivery: same DC, same SKU, same supplier_ref, same date+time — that
# combination repeating usually means the same stock movement got logged
# twice (e.g. a retry in the feeding system), inflating stock-on-hand.

dupe_pk = con.sql(f"""
    SELECT movement_id, COUNT(*) AS n FROM {raw}
    GROUP BY movement_id HAVING COUNT(*) > 1
""").df()
print(f"Duplicate movement_id: {len(dupe_pk)}")
print(dupe_pk)

dupe_movement = con.sql(f"""
    SELECT dc_code, sku, supplier_ref, movement_date, movement_time, COUNT(*) AS n
    FROM {raw}
    GROUP BY dc_code, sku, supplier_ref, movement_date, movement_time
    HAVING COUNT(*) > 1
""").df()
print(f"\nExact duplicate movements (same DC/sku/supplier_ref/date/time): {len(dupe_movement)}")
print(dupe_movement.head(20))

Duplicate movement_id: 0
Empty DataFrame
Columns: [movement_id, n]
Index: []

Exact duplicate movements (same DC/sku/supplier_ref/date/time): 0
Empty DataFrame
Columns: [dc_code, sku, supplier_ref, movement_date, movement_time, n]
Index: []


In [8]:
# Referential integrity: supplier_ref against suppliers.csv

# WHY: supplier_ref embeds a supplier_code (e.g. 'SUP003' inside
# 'SUP003-DEL-42918') rather than storing it as its own column. Pulling
# that prefix out and checking it against suppliers.csv confirms every
# delivery is traceable to a real, known supplier.

orphan_suppliers = con.sql(f"""
    SELECT m.movement_id, m.supplier_ref,
           regexp_extract(m.supplier_ref, '^(SUP\\d+)', 1) AS extracted_supplier_code
    FROM {raw} m
    LEFT JOIN read_csv_auto('{suppliers_path}') s
           ON regexp_extract(m.supplier_ref, '^(SUP\\d+)', 1) = s.supplier_code
    WHERE s.supplier_code IS NULL
""").df()
print(f"supplier_ref values with no matching supplier_code in suppliers.csv: {len(orphan_suppliers)}")
print(orphan_suppliers.head(10))

supplier_ref values with no matching supplier_code in suppliers.csv: 0
Empty DataFrame
Columns: [movement_id, supplier_ref, extracted_supplier_code]
Index: []


In [9]:
# movement_type and dc_code: categorical validity

# WHY: shows every distinct value and its count so you can eyeball for
# typos/case issues, and gives a sense of whether DC volume is evenly
# spread or lopsided (a DC with far fewer movements might mean a data
# feed problem for that specific site, not fewer real deliveries).

print("=== movement_type ===")
print(con.sql(f"SELECT movement_type, COUNT(*) AS n FROM {raw} GROUP BY movement_type ORDER BY n DESC").df())

print("\n=== dc_code ===")
print(con.sql(f"SELECT dc_code, COUNT(*) AS n FROM {raw} GROUP BY dc_code ORDER BY n DESC").df())

=== movement_type ===
  movement_type     n
0       receipt  6718
1     write_off  2274
2      transfer  2179

=== dc_code ===
    dc_code     n
0  DC-LEEDS  1918
1   DC-BRIS  1877
2   DC-MANC  1871
3   DC-BHAM  1870
4   DC-NORW  1840
5   DC-GLAS  1795


In [10]:
# quantity validity and outliers

# WHY: quantity should always be positive (confirmed: zero negative/zero
# values exist). IQR then catches entry errors that pass the basic
# positivity check but are still implausible — e.g. a single receipt of
# 50,000 units when the typical receipt is 10-500.

bad_qty = con.sql(f"SELECT * FROM {raw} WHERE quantity <= 0").df()
print(f"quantity <= 0: {len(bad_qty)}")

outliers = con.sql(f"""
    WITH stats AS (
        SELECT QUANTILE_CONT(quantity, 0.25) AS q1,
               QUANTILE_CONT(quantity, 0.75) AS q3
        FROM {raw}
    )
    SELECT m.movement_id, m.movement_type, m.quantity
    FROM {raw} m, stats
    WHERE m.quantity < (stats.q1 - 1.5*(stats.q3-stats.q1))
       OR m.quantity > (stats.q3 + 1.5*(stats.q3-stats.q1))
    ORDER BY m.quantity DESC
""").df()
print(f"\nquantity outliers (IQR): {len(outliers)}")
print(outliers.head(15))

quantity <= 0: 0

quantity outliers (IQR): 0
Empty DataFrame
Columns: [movement_id, movement_type, quantity]
Index: []


In [11]:
# sku format consistency

# WHY: sku should always follow NP-SKU-##### — a malformed sku breaks
# any join back to a products table.

bad_sku = con.sql(f"SELECT * FROM {raw} WHERE sku !~ '^NP-SKU-\\d{{5}}$'").df()
print(f"Malformed sku: {len(bad_sku)}")
print(bad_sku.head(10))

Malformed sku: 0
Empty DataFrame
Columns: [movement_id, dc_code, sku, movement_type, quantity, supplier_ref, movement_date, movement_time, timezone, reached_epos, week_number, notes]
Index: []


In [13]:
# supplier_ref format consistency

# WHY: supplier_ref should always follow SUP###-DEL-###### — this is
# separate from Cell 3 (which checked the SUPxxx part is a REAL
# supplier); this checks the whole reference is correctly SHAPED in the
# first place, since a malformed one could still accidentally match a
# real supplier_code by coincidence.

bad_ref = con.sql(f"SELECT * FROM {raw} WHERE supplier_ref !~ '^SUP\\d{{3}}-DEL-\\d+$'").df()
print(f"Malformed supplier_ref: {len(bad_ref)}")
print(bad_ref.head(10))

Malformed supplier_ref: 0
Empty DataFrame
Columns: [movement_id, dc_code, sku, movement_type, quantity, supplier_ref, movement_date, movement_time, timezone, reached_epos, week_number, notes]
Index: []


In [19]:
# WHY: one single, clean summary table — every movement_date row gets
# exactly ONE label describing its format, then we count rows per label.
# This combines the "what shape is it" check and the "can we tell DD/MM
# from MM/DD" check into one pass, so you get one readable table instead
# of several scattered prints.

date_format_summary = con.sql(f"""
    SELECT
        CASE
            WHEN movement_date ~ '^\\d{{4}}-\\d{{2}}-\\d{{2}}$'
                THEN 'ISO (YYYY-MM-DD)'

            WHEN movement_date ~ '^\\d{{2}}/\\d{{2}}/\\d{{4}}$'
                 AND CAST(regexp_extract(movement_date, '^(\\d{{2}})/\\d{{2}}/\\d{{4}}$', 1) AS INTEGER) > 12
                THEN 'Slash, confirmed DD/MM/YYYY'

            WHEN movement_date ~ '^\\d{{2}}/\\d{{2}}/\\d{{4}}$'
                 AND CAST(regexp_extract(movement_date, '^\\d{{2}}/(\\d{{2}})/\\d{{4}}$', 1) AS INTEGER) > 12
                THEN 'Slash, confirmed MM/DD/YYYY'

            WHEN movement_date ~ '^\\d{{2}}/\\d{{2}}/\\d{{4}}$'
                THEN 'Slash, ambiguous (both parts <=12)'

            ELSE 'Unrecognized format'
        END AS date_format,
        COUNT(*) AS row_count,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_of_total
    FROM {raw}
    GROUP BY date_format
    ORDER BY row_count DESC
""").df()

print(date_format_summary.to_string(index=False))

                       date_format  row_count  pct_of_total
                  ISO (YYYY-MM-DD)       5588          50.0
       Slash, confirmed DD/MM/YYYY       3425          30.7
Slash, ambiguous (both parts <=12)       2158          19.3


In [14]:
# movement_date: explicit parsing + sanity checks

# WHY: this is the key one for this table. We parse movement_date with an
# EXPLICIT format string (%d/%m/%Y) instead of letting DuckDB guess, so
# the 2,158 ambiguous rows are read correctly instead of silently wrong.
# Then we check for dates in the future or implausibly old.

bad_dates = con.sql(f"""
    SELECT movement_id, movement_date,
           TRY_STRPTIME(movement_date, '%d/%m/%Y') AS parsed_date
    FROM {raw}
    WHERE TRY_STRPTIME(movement_date, '%d/%m/%Y') IS NULL
""").df()
print(f"movement_date values that fail to parse as DD/MM/YYYY: {len(bad_dates)}")
print(bad_dates.head(10))

future_dates = con.sql(f"""
    SELECT movement_id, movement_date
    FROM {raw}
    WHERE TRY_STRPTIME(movement_date, '%d/%m/%Y') > CURRENT_DATE
""").df()
print(f"\nmovement_date in the future: {len(future_dates)}")
print(future_dates.head(10))

movement_date values that fail to parse as DD/MM/YYYY: 5588
   movement_id movement_date parsed_date
0            1    2025-01-04         NaT
1            2    2025-01-06         NaT
2            3    2025-01-05         NaT
3            4    2025-01-02         NaT
4            5    2025-01-03         NaT
5            6    2025-01-05         NaT
6            7    2025-01-02         NaT
7            8    2025-01-01         NaT
8            9    2025-01-01         NaT
9           10    2025-01-02         NaT

movement_date in the future: 0
Empty DataFrame
Columns: [movement_id, movement_date]
Index: []


In [24]:
# week_number vs movement_date: do they actually agree?

# WHY: week_number should be the ISO calendar week of movement_date. If
# the two disagree, something generated them independently and
# inconsistently (e.g. one source computed week_number before a late
# correction was made to movement_date).

week_mismatch = con.sql(f"""
    SELECT movement_id, movement_date, week_number,
           EXTRACT(WEEK FROM TRY_STRPTIME(movement_date, '%d/%m/%Y')) AS calculated_week
    FROM {raw}
    WHERE week_number != EXTRACT(WEEK FROM TRY_STRPTIME(movement_date, '%d/%m/%Y'))
""").df()
print(f"week_number not matching the Slash week of movement_date: {len(week_mismatch)}")
print(week_mismatch.head(10))

week_number not matching the Slash week of movement_date: 1566
   movement_id movement_date  week_number  calculated_week
0         5590    07/07/2025           27               28
1         5591    07/07/2025           27               28
2         5593    07/07/2025           27               28
3         5601    07/07/2025           27               28
4         5602    08/07/2025           27               28
5         5604    07/07/2025           27               28
6         5614    08/07/2025           27               28
7         5617    07/07/2025           27               28
8         5618    08/07/2025           27               28
9         5622    08/07/2025           27               28


In [21]:
# week_number vs movement_date: do they actually agree?

# WHY: week_number should be the ISO calendar week of movement_date. If
# the two disagree, something generated them independently and
# inconsistently (e.g. one source computed week_number before a late
# correction was made to movement_date).

week_mismatch = con.sql(f"""
    SELECT movement_id, movement_date, week_number,
           EXTRACT(WEEK FROM TRY_STRPTIME(movement_date, '%Y-%m-%d')) AS calculated_week
    FROM {raw}
    WHERE week_number != EXTRACT(WEEK FROM TRY_STRPTIME(movement_date, '%Y-%m-%d'))
""").df()
print(f"week_number not matching the ISO week of movement_date: {len(week_mismatch)}")
print(week_mismatch.head(10))

week_number not matching the ISO week of movement_date: 1639
   movement_id movement_date  week_number  calculated_week
0            2    2025-01-06            1                2
1           14    2025-01-06            1                2
2           17    2025-01-07            1                2
3           18    2025-01-07            1                2
4           19    2025-01-06            1                2
5           20    2025-01-06            1                2
6           22    2025-01-06            1                2
7           24    2025-01-06            1                2
8           27    2025-01-07            1                2
9           31    2025-01-07            1                2


In [23]:
# week_number vs movement_date: do they actually agree?

# WHY: week_number should be the ISO calendar week of movement_date. The
# ORIGINAL versions of this check only tried parsing as DD/MM/YYYY — which
# means every ISO-format row (2025-01-04 style, ~50% of the file) failed
# to parse, came back NULL, and SQL's "NULL != anything" is never TRUE —
# so those rows were silently DROPPED from the mismatch check instead of
# being tested. That would have hidden real problems in half the file.
#
# Fixed version: try ISO first, fall back to DD/MM/YYYY (COALESCE picks
# whichever one actually parses) — so every row gets a real date and a
# real comparison, regardless of which format it's written in.

week_mismatch = con.sql(f"""
    SELECT
        movement_id,
        movement_date,
        week_number,
        COALESCE(
            TRY_STRPTIME(movement_date, '%Y-%m-%d'),
            TRY_STRPTIME(movement_date, '%d/%m/%Y')
        ) AS parsed_date,
        EXTRACT(WEEK FROM COALESCE(
            TRY_STRPTIME(movement_date, '%Y-%m-%d'),
            TRY_STRPTIME(movement_date, '%d/%m/%Y')
        )) AS calculated_week
    FROM {raw}
    WHERE week_number != EXTRACT(WEEK FROM COALESCE(
        TRY_STRPTIME(movement_date, '%Y-%m-%d'),
        TRY_STRPTIME(movement_date, '%d/%m/%Y')
    ))
""").df()

print(f"week_number not matching of movement_date: {len(week_mismatch)}")
print(week_mismatch.head(10))

week_number not matching of movement_date: 3205
   movement_id movement_date  week_number parsed_date  calculated_week
0            2    2025-01-06            1  2025-01-06                2
1           14    2025-01-06            1  2025-01-06                2
2           17    2025-01-07            1  2025-01-07                2
3           18    2025-01-07            1  2025-01-07                2
4           19    2025-01-06            1  2025-01-06                2
5           20    2025-01-06            1  2025-01-06                2
6           22    2025-01-06            1  2025-01-06                2
7           24    2025-01-06            1  2025-01-06                2
8           27    2025-01-07            1  2025-01-07                2
9           31    2025-01-07            1  2025-01-07                2


In [22]:
# timezone vs month: rough BST/UTC sense check

# WHY: the UK uses BST (UTC+1) roughly late March to late October, and
# UTC the rest of the year. This isn't an exact day-level rule (the exact
# switchover date varies each year), but a timezone value that's wildly
# inconsistent with the month (e.g. BST showing up in January) is worth a
# look. This is a SENSE CHECK, not a hard pass/fail rule.

tz_by_month = con.sql(f"""
    SELECT
        EXTRACT(MONTH FROM COALESCE(
            TRY_STRPTIME(movement_date, '%Y-%m-%d'),
            TRY_STRPTIME(movement_date, '%d/%m/%Y')
        )) AS month,
        timezone,
        COUNT(*) AS n
    FROM {raw}
    GROUP BY month, timezone
    ORDER BY month, timezone
""").df()

print("=== timezone value by month ===")
print(tz_by_month.to_string(index=False))
print("\nExpect UTC to dominate Nov-Mar, BST to dominate Apr-Oct. Anything inverted is worth checking.")

=== timezone value by month ===
 month timezone   n
     1      UTC 992
     2      UTC 832
     3      UTC 930
     4      UTC 959
     5      UTC 897
     6      UTC 945
     7      BST  42
     7      UTC 913
     8      BST 608
     8      UTC 350
     9      BST 617
     9      UTC 332
    10      BST 669
    10      UTC 327
    11      BST 646
    11      UTC 276
    12      BST 548
    12      UTC 288

Expect UTC to dominate Nov-Mar, BST to dominate Apr-Oct. Anything inverted is worth checking.


In [25]:
# reached_epos: what does blank actually mean?

# WHY: reached_epos only ever contains 'Y' or blank — there's no 'N'.
# This cell doesn't assume which interpretation is correct; it just shows
# you the breakdown by movement_type and dc_code so you (or whoever owns
# this table) can decide: is blank a real business state ("hasn't synced
# to EPOS yet") or just a data gap?

epos_breakdown = con.sql(f"""
    SELECT movement_type, reached_epos, COUNT(*) AS n
    FROM {raw}
    GROUP BY movement_type, reached_epos
    ORDER BY movement_type, reached_epos
""").df()
print("=== reached_epos breakdown by movement_type ===")
print(epos_breakdown)

=== reached_epos breakdown by movement_type ===
  movement_type reached_epos     n
0       receipt            Y  6658
1       receipt          NaN    60
2      transfer            Y  2155
3      transfer          NaN    24
4     write_off            Y  2257
5     write_off          NaN    17


In [29]:
# ── Format Consistency Audit: same value, inconsistent spelling/casing? ──

text_columns = ['movement_type', 'timezone', 'reached_epos']

consistency_report = []
for col in text_columns:
    result = con.sql(f"""
        SELECT
            '{col}' AS column_name,
            COUNT(DISTINCT {col}) AS raw_distinct_count,
            COUNT(DISTINCT LOWER(TRIM({col}))) AS normalized_distinct_count
        FROM read_csv_auto('{path}')
    """).df()
    consistency_report.append(result)

import pandas as pd
consistency_df = pd.concat(consistency_report, ignore_index=True)

# If raw > normalized, some values are duplicates of each other under
# different casing/spacing — this column shows exactly how many.
consistency_df['extra_variants'] = (
    consistency_df['raw_distinct_count'] - consistency_df['normalized_distinct_count']
)
consistency_df

,column_name,raw_distinct_count,normalized_distinct_count,extra_variants
0,movement_type,3,3,0
1,timezone,2,2,0
2,reached_epos,1,1,0


In [31]:
movement_type = con.sql(f"""
    SELECT movement_type, COUNT(*) AS n
    FROM read_csv_auto('{path}')
    GROUP BY movement_type
    ORDER BY n DESC
""").df()

movement_type

,movement_type,n
0,receipt,6718
1,write_off,2274
2,transfer,2179


In [27]:
# structural limitation: transfers have no direction (no code — read and note this)

# WHY: not something a query can fix. 'transfer' movements only record
# ONE dc_code — there's no from_dc/to_dc pair, so you cannot tell whether
# a given transfer row means stock LEFT that DC or ARRIVED at it. Any
# stock-on-hand calculation that tries to net receipt - write_off +/- transfer
# per DC will be wrong for the transfer component specifically, because the
# sign is unknowable from this table alone.

transfer_count = con.sql(f"SELECT COUNT(*) FROM {raw} WHERE movement_type = 'transfer'").fetchone()[0]
print(f"'transfer' rows with no direction field: {transfer_count}")
print("This is a data MODEL gap, not a data quality bug — flag it to whoever owns this table's schema. "
      "A stock reconciliation built on this file cannot currently net transfers correctly.")

'transfer' rows with no direction field: 2179
This is a data MODEL gap, not a data quality bug — flag it to whoever owns this table's schema. A stock reconciliation built on this file cannot currently net transfers correctly.


In [28]:
# Row volume by month

monthly_volume = con.sql(f"""
    SELECT
        strftime(COALESCE(
            TRY_STRPTIME(movement_date, '%Y-%m-%d'),
            TRY_STRPTIME(movement_date, '%d/%m/%Y')
        ), '%Y-%m') AS month,
        COUNT(*) AS n_rows
    FROM {raw}
    GROUP BY month
    ORDER BY month
""").df()
print("=== Row volume by month ===")
print(monthly_volume.to_string(index=False))

latest = con.sql(f"""
    SELECT MAX(COALESCE(
        TRY_STRPTIME(movement_date, '%Y-%m-%d'),
        TRY_STRPTIME(movement_date, '%d/%m/%Y')
    )) AS latest_movement_date
    FROM {raw}
""").df()
print("\n=== Latest movement_date in the file ===")
print(latest.to_string(index=False))
print("\nCompare this to today's date (2026-10-02) — check whether the gap is still what we thought once both date formats are counted.")

=== Row volume by month ===
  month  n_rows
2025-01     992
2025-02     832
2025-03     930
2025-04     959
2025-05     897
2025-06     945
2025-07     955
2025-08     958
2025-09     949
2025-10     996
2025-11     922
2025-12     836

=== Latest movement_date in the file ===
latest_movement_date
          2025-12-30

Compare this to today's date (2026-10-02) — check whether the gap is still what we thought once both date formats are counted.


In [34]:
# Format consistency: Flag columns that contain mixed data types
def format_consistency_check(con, path):
    # Step 1: get every column name in the file
    schema = con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").df()
    columns = schema['column_name'].tolist()

    all_rows = []
    for col in columns:
        # Step 2: for THIS column, classify every value into a bucket
        breakdown = con.sql(f"""
            SELECT
                '{col}' AS column_name,
                CASE
                    WHEN "{col}" IS NULL OR TRIM("{col}") = '' THEN 'NULL/EMPTY'
                    WHEN TRY_CAST("{col}" AS BIGINT) IS NOT NULL THEN 'INTEGER'
                    WHEN TRY_CAST("{col}" AS DOUBLE) IS NOT NULL THEN 'DECIMAL'
                    WHEN TRY_CAST("{col}" AS DATE) IS NOT NULL THEN 'DATE'
                    WHEN TRY_CAST("{col}" AS TIMESTAMP) IS NOT NULL THEN 'TIMESTAMP'
                    WHEN lower("{col}") IN ('true','false') THEN 'BOOLEAN'
                    ELSE 'TEXT'
                END AS detected_type,
                COUNT(*) AS row_count
            FROM read_csv_auto('{path}', all_varchar=True)
            GROUP BY detected_type
            ORDER BY row_count DESC
        """).df()
        all_rows.append(breakdown)

    result = pd.concat(all_rows, ignore_index=True)

    # Step 3: flag a column only if it has 2+ real (non-null) types
    n_types = result[result['detected_type'] != 'NULL/EMPTY'].groupby('column_name')['detected_type'].transform('nunique')
    result['is_mixed'] = (n_types.reindex(result.index).fillna(0) > 1)
    return result

report = format_consistency_check(con, path)
report                       # full breakdown — bare, no print(), so it renders as a table

,column_name,detected_type,row_count,is_mixed
0,movement_id,INTEGER,11171,False
1,dc_code,TEXT,11171,False
2,sku,TEXT,11171,False
3,movement_type,TEXT,11171,False
4,quantity,INTEGER,11171,False
5,supplier_ref,TEXT,11171,False
6,movement_date,DATE,5588,True
7,movement_date,TEXT,5583,True
8,movement_time,TEXT,11171,False
9,timezone,TEXT,11171,False
